# TinyOCT v2 — Google Colab Training

**Architecture:** MobileNetV3-Small + Multi-scale LaplacianLayer + RLAPv3 (H/V/Focal streams) + PrototypeHead  
**Target GPU:** Tesla T4 (15 GB VRAM) — batch_size=64, orient_loss enabled  
**Dataset:** OCT2017 (Kermany 2018) — 4 classes: CNV / DME / DRUSEN / NORMAL

### Run order
1. Fill in **Section 0** (repo URL + optional W&B key)
2. Run all cells top-to-bottom (`Runtime → Run all`)
3. Monitor training in the output of **Section 5** or in W&B
4. Final checkpoint is auto-saved to `MyDrive/tinyoct/checkpoints/`

---
## Section 0 — User Configuration
**Edit this cell before running anything else.**

In [ ]:
# ── HOW TO GET THE CODEBASE INTO COLAB ───────────────────────────────────────
# Option A (recommended): GitHub
#   Push your local repo, then paste the URL below.
#   For a private repo, use: https://<TOKEN>@github.com/user/tinyoct.git
GITHUB_REPO_URL = ""  # e.g. "https://github.com/yourusername/tinyoct.git"

# Option B: Google Drive zip
#   1. Locally: zip -r tinyoct_code.zip tinyoct/ --exclude='*/.git/*' --exclude='*/__pycache__/*' --exclude='*/.venv/*' --exclude='*/data/*' --exclude='*/checkpoints/*'
#   2. Upload to Google Drive → right-click → Share → copy FILE ID from URL
#   3. Paste FILE_ID below and leave GITHUB_REPO_URL empty.
GDRIVE_CODE_FILE_ID = ""  # e.g. "1BxiMVs0XRA5nFMdKvBdBZjgmUUqptlbs74OgVE2upms"

# ── DATASET PATH ─────────────────────────────────────────────────────────────
# Leave as None to auto-detect from common Colab paths.
# Override if your layout is different, e.g. "/content/data/OCT2017"
OCT2017_PATH_OVERRIDE = None

# ── WEIGHTS & BIASES (optional) ───────────────────────────────────────────────
# Leave blank to disable W&B (results printed to notebook output instead).
WANDB_API_KEY = ""  # paste your key from https://wandb.ai/authorize
WANDB_PROJECT  = "tinyoct"

# ── TRAINING MODE ─────────────────────────────────────────────────────────────
# SMOKE_TEST_ONLY = True  → run 3 epochs to verify the pipeline, then stop.
# SMOKE_TEST_ONLY = False → run full 30-epoch training.
SMOKE_TEST_ONLY = False

print("Configuration loaded. Proceed to Section 1.")

---
## Section 1 — Environment Setup

In [12]:
# 1a. Verify GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to GPU (Runtime → Change runtime type → T4 GPU)"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}")

Tesla T4, 15360 MiB, 580.82.07
PyTorch 2.10.0+cu128 | CUDA 12.8 | GPU: Tesla T4


In [13]:
# 1b. Install Python dependencies
# timm: backbone (MobileNetV3-Small)
# omegaconf: config loading with dot-path overrides
# wandb: experiment tracking (optional)
# scikit-learn: metrics (F1, AUC, ECE)
# medmnist: OCTMNIST dataset loader
!pip install timm omegaconf wandb scikit-learn gdown medmnist --quiet
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.5 MB/s eta 0:00:00
Dependencies installed.


In [14]:
# 1c. Mount Google Drive (for checkpoint saving + optional code download)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/tinyoct/checkpoints', exist_ok=True)
print("Drive mounted. Checkpoints will be saved to: /content/drive/MyDrive/tinyoct/checkpoints/")

Mounted at /content/drive
Drive mounted. Checkpoints will be saved to: /content/drive/MyDrive/tinyoct/checkpoints/


---
## Section 2 — Get Codebase

In [17]:
import os, sys, subprocess

TINYOCT_DIR = '/content/tinyoct'
GITHUB_REPO_URL="https://github.com/mazleon/tinyoct.git"

if os.path.isdir(TINYOCT_DIR) and os.path.exists(f'{TINYOCT_DIR}/tinyoct/models/tinyoct.py'):
    print(f"Codebase already present at {TINYOCT_DIR} — skipping download.")

elif GITHUB_REPO_URL:
    print(f"Cloning from GitHub: {GITHUB_REPO_URL}")
    result = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_REPO_URL, TINYOCT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")
    print("Clone complete.")

elif GDRIVE_CODE_FILE_ID:
    print(f"Downloading codebase from Google Drive (file ID: {GDRIVE_CODE_FILE_ID})...")
    import gdown
    zip_path = '/content/tinyoct_code.zip'
    gdown.download(id=GDRIVE_CODE_FILE_ID, output=zip_path, quiet=False)
    print("Extracting...")
    os.makedirs(TINYOCT_DIR, exist_ok=True)
    result = subprocess.run(
        ['unzip', '-q', '-o', zip_path, '-d', TINYOCT_DIR],
        capture_output=True, text=True
    )
    # Handle case where zip contains a single top-level folder
    extracted = [d for d in os.listdir(TINYOCT_DIR)
                 if os.path.isdir(os.path.join(TINYOCT_DIR, d))]
    if len(extracted) == 1 and not os.path.exists(f'{TINYOCT_DIR}/tinyoct'):
        inner = os.path.join(TINYOCT_DIR, extracted[0])
        os.system(f'mv {inner}/* {TINYOCT_DIR}/ && rmdir {inner}')
    print("Extraction complete.")

else:
    raise ValueError(
        "No codebase source configured.\n"
        "Set GITHUB_REPO_URL or GDRIVE_CODE_FILE_ID in Section 0."
    )

# Add to Python path
if TINYOCT_DIR not in sys.path:
    sys.path.insert(0, TINYOCT_DIR)

# Verify key files are present
required = [
    'tinyoct/models/tinyoct.py',
    'tinyoct/models/rlap.py',
    'tinyoct/losses/focal_loss.py',
    'scripts/train.py',
    'configs/base.yaml',
]
missing = [f for f in required if not os.path.exists(os.path.join(TINYOCT_DIR, f))]
if missing:
    raise FileNotFoundError(f"Codebase incomplete. Missing files: {missing}")

print(f"\nCodebase verified at {TINYOCT_DIR}")
print("Files:", os.listdir(TINYOCT_DIR))

Cloning from GitHub: https://github.com/mazleon/tinyoct.git
Clone complete.


FileNotFoundError: Codebase incomplete. Missing files: ['tinyoct/losses/focal_loss.py']

---
## Section 3 — Dataset Setup

In [18]:
import os

def find_oct2017_root():
    """Find the OCT2017 directory by looking for the train/CNV subfolder."""
    # Search candidates in order of likelihood
    candidates = [
        OCT2017_PATH_OVERRIDE,          # user override from Section 0
        '/content/data/OCT2017',
        '/content/data/oct2017',
        '/content/data/OCT',
        '/content/OCT2017',
        '/content/data',
    ]
    for path in candidates:
        if path and os.path.isdir(path):
            # Validate: must contain train/CNV or train/cnv
            for sub in ['train/CNV', 'train/cnv', 'train']:
                if os.path.isdir(os.path.join(path, sub)):
                    return path
    return None

OCT2017_PATH = find_oct2017_root()

if OCT2017_PATH is None:
    # Show what's in /content/data to help debug
    print("OCT2017 not found. Contents of /content/data:")
    if os.path.exists('/content/data'):
        for root, dirs, files in os.walk('/content/data'):
            level = root.replace('/content/data', '').count(os.sep)
            if level < 3:
                indent = '  ' * level
                print(f'{indent}{os.path.basename(root)}/')
    raise FileNotFoundError(
        "Cannot find OCT2017 dataset.\n"
        "Set OCT2017_PATH_OVERRIDE in Section 0 to the folder containing train/test/val."
    )

# Count samples per split
print(f"OCT2017 found at: {OCT2017_PATH}")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(OCT2017_PATH, split)
    if os.path.isdir(split_path):
        total = sum(len(files) for _, _, files in os.walk(split_path))
        classes = sorted(os.listdir(split_path))
        print(f"  {split:5s}: {total:6,} images  |  classes: {classes}")

NameError: name 'OCT2017_PATH_OVERRIDE' is not defined

---
## Section 4 — Colab Config

In [19]:
# Write the Colab experiment config directly into the repo.
# This patches data paths and hardware settings for T4 over base.yaml.

import yaml, os

# Determine W&B setting
use_wandb = bool(WANDB_API_KEY.strip())

colab_cfg = {
    'defaults': ['base'],
    'project': {
        'name': 'tinyoct-colab',
    },
    'data': {
        'oct2017_path': OCT2017_PATH,
        'num_workers': 2,   # Colab free: 2 CPUs; change to 4 if on Colab Pro
    },
    'train': {
        'epochs': 3 if SMOKE_TEST_ONLY else 30,
        'batch_size': 64,   # T4 15 GB: safe with orient double-forward
        'num_workers': 2,
        'lr': 1.0e-3,
        'warmup_epochs': 0 if SMOKE_TEST_ONLY else 2,
    },
    'model': {
        'rlap': {
            'focal_spot': True,  # TinyOCT v2: FocalSpotStream for DRUSEN
        }
    },
    'checkpoint': {
        'dir': '/content/drive/MyDrive/tinyoct/checkpoints',
        'save_best': True,
        'save_every_epoch': False,
        'monitor': 'drusen_f1',
        'mode': 'max',
    },
    'logging': {
        'use_wandb': use_wandb,
        'wandb_project': WANDB_PROJECT,
        'output_dir': '/content/outputs',
    },
}

cfg_path = os.path.join(TINYOCT_DIR, 'configs/experiment_colab.yaml')
with open(cfg_path, 'w') as f:
    yaml.dump(colab_cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {cfg_path}")
print(f"  Epochs:        {colab_cfg['train']['epochs']}")
print(f"  Batch size:    {colab_cfg['train']['batch_size']}")
print(f"  Data path:     {OCT2017_PATH}")
print(f"  Checkpoint:    {colab_cfg['checkpoint']['dir']}")
print(f"  W&B logging:   {use_wandb}")
print(f"  FocalSpotStream (v2): True")

os.makedirs('/content/outputs', exist_ok=True)

NameError: name 'WANDB_API_KEY' is not defined

---
## Section 4b — W&B Login (optional)

In [ ]:
if WANDB_API_KEY.strip():
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print(f"W&B logged in. Runs will appear at: https://wandb.ai/{WANDB_PROJECT}")
else:
    import os
    os.environ['WANDB_MODE'] = 'disabled'
    print("W&B disabled. Metrics will be printed to notebook output only.")

---
## Section 5 — Quick Sanity Check (unit tests)

In [ ]:
# Run the unit test suite to verify all components are working correctly
# on this Colab environment before spending 30 epochs on training.
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_model.py', '-v', '--tb=short', '-q'],
    cwd=TINYOCT_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Unit tests failed — do not proceed to training. Fix the errors above.")
print("All tests passed. Safe to proceed to training.")

---
## Section 6 — Training

In [ ]:
import subprocess, sys, os

mode = "SMOKE TEST (3 epochs)" if SMOKE_TEST_ONLY else "FULL TRAINING (30 epochs)"
print(f"{'='*60}")
print(f"  TinyOCT v2 — {mode}")
print(f"  Config: configs/experiment_colab.yaml")
print(f"  GPU:    {torch.cuda.get_device_name(0)}")
print(f"{'='*60}\n")

cmd = [
    sys.executable, '-u',
    'scripts/train.py',
    '--config', 'configs/experiment_colab.yaml',
]

# Stream output line by line so progress is visible in the notebook
process = subprocess.Popen(
    cmd,
    cwd=TINYOCT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'},
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    process.terminate()
    print("\nTraining interrupted by user.")

if process.returncode not in (0, None, -15):
    raise RuntimeError(f"Training exited with code {process.returncode}")
else:
    print(f"\nTraining complete. Best checkpoint saved to Drive.")

---
## Section 7 — Evaluation

In [ ]:
import os, glob

# Find best checkpoint (Drive first, then local fallback)
drive_ckpts = glob.glob('/content/drive/MyDrive/tinyoct/checkpoints/best*.pth')
local_ckpts = glob.glob(os.path.join(TINYOCT_DIR, 'checkpoints/**/*.pth'), recursive=True)
all_ckpts   = drive_ckpts + local_ckpts

if not all_ckpts:
    raise FileNotFoundError(
        "No checkpoint found. Run Section 6 first, or upload a checkpoint to "
        "/content/drive/MyDrive/tinyoct/checkpoints/"
    )

BEST_CKPT = sorted(all_ckpts, key=os.path.getmtime)[-1]
print(f"Using checkpoint: {BEST_CKPT}")

In [ ]:
import subprocess, sys, os

preds_path = '/content/outputs/predictions.npz'

cmd = [
    sys.executable, '-u',
    'scripts/evaluate.py',
    '--checkpoint', BEST_CKPT,
    '--config',     'configs/experiment_colab.yaml',
    '--calibrate',
    '--save-preds', preds_path,
]

process = subprocess.Popen(
    cmd,
    cwd=TINYOCT_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'},
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if os.path.exists(preds_path):
    print(f"\nPredictions saved to: {preds_path}")

In [ ]:
# Generate ROC curves from real model predictions
import subprocess, sys, os

preds_path = '/content/outputs/predictions.npz'
roc_out    = '/content/outputs/roc_curve.png'

if os.path.exists(preds_path):
    result = subprocess.run(
        [sys.executable, 'scripts/generate_roc.py',
         '--preds', preds_path, '--out', roc_out],
        cwd=TINYOCT_DIR, capture_output=True, text=True
    )
    print(result.stdout or result.stderr)

    if os.path.exists(roc_out):
        from IPython.display import Image, display
        display(Image(roc_out))
else:
    print("No predictions file found — run the evaluation cell first.")

---
## Section 8 — Run Individual Ablation (optional)
Use this to run a single ablation row (R0–R9) for the paper Table 2.

In [ ]:
# Set to the ablation key you want to run, e.g. "R6_focal_stream"
ABLATION_KEY = "R9_full_v2"

import subprocess, sys, os

print(f"Running ablation: {ABLATION_KEY}")
cmd = [
    sys.executable, '-u',
    'scripts/train.py',
    '--config',   'configs/experiment_colab.yaml',
    '--ablation', ABLATION_KEY,
]
process = subprocess.Popen(
    cmd,
    cwd=TINYOCT_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'},
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    process.terminate()
    print("\nAblation interrupted.")

---
## Section 9 — Save All Outputs to Drive

In [ ]:
import os, shutil, glob
from datetime import datetime

DRIVE_OUT = '/content/drive/MyDrive/tinyoct'
os.makedirs(DRIVE_OUT, exist_ok=True)

saved = []

# 1. Figures / outputs
for pattern in ['/content/outputs/*.png', '/content/outputs/*.npz', '/content/outputs/*.json']:
    for f in glob.glob(pattern):
        dest = os.path.join(DRIVE_OUT, 'outputs', os.path.basename(f))
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(f, dest)
        saved.append(dest)

# 2. Paper figures from codebase
figures_src = os.path.join(TINYOCT_DIR, 'paper', 'figures')
if os.path.isdir(figures_src):
    for f in glob.glob(os.path.join(figures_src, '*.png')):
        dest = os.path.join(DRIVE_OUT, 'paper_figures', os.path.basename(f))
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(f, dest)
        saved.append(dest)

# 3. Training logs (W&B offline, if any)
for logfile in glob.glob(os.path.join(TINYOCT_DIR, 'outputs/*.log')):
    dest = os.path.join(DRIVE_OUT, 'logs', os.path.basename(logfile))
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(logfile, dest)
    saved.append(dest)

print(f"Saved {len(saved)} files to Google Drive ({DRIVE_OUT}):")
for f in saved:
    print(f"  {f}")
if not saved:
    print("  (nothing to save yet — run training and evaluation first)")